# O‘zMSt 285:2024, 1-algoritm: tavsiyalarning tezlikka ta’siri

Taqqoslanadigan variantlar (har ikkala rejimda, p = 256 va p = 16):

| Belgi | Variant |
|---|---|
| `std` | standart algoritm (BaytZichlash) |
| `std+toq` | standart + toq kₑ (2-tavsiya) |
| `std+S` | standart + raund ichidagi S-box qatlami (3-tavsiya) |
| `sbz` | SBZ (1-tavsiya) |
| `sbz+toq` | SBZ + toq kₑ |
| `sbz+toq+S` | uchala tavsiya birgalikda |

Tezlik uch xil usulda baholanadi:
1. **C realizatsiyasi** (gcc -O2) — eng ishonchli o‘lchov; kompyuterda C kompilyatori (gcc, clang yoki MinGW) bo‘lsa ishlaydi.
2. **Python (numpy)** realizatsiyasi — nisbiy taqqoslash uchun.
3. **Amallar sonining modeli** — platformaga bog‘liq bo‘lmagan baho.

Daftar mustaqil: modullar 0-hujayrada yuklanadi. Faqat `numpy` kerak (grafik uchun `matplotlib` ixtiyoriy).

In [1]:
# 0-hujayra. Dastur modullarini yuklash (daftar mustaqil ishlaydi).
# Agar modullar shu papkada bo'lsa, ular ishlatiladi; bo'lmasa quyidagi matndan xotirada yaratiladi.
import sys, types, importlib.util
_MODULLAR = {
    'algoritm': '"""\nO‘zMSt 285:2024, 1-algoritm: vektorlashtirilgan realizatsiya.\nVariantlar: p = 256 yoki p = 16; yutish o‘zgartirishi bz = \'std\' (BaytZichlash) yoki \'sbz\' (S(h ⊕ S(n))).\nBarcha funksiyalar (N, 32) o‘lchamli numpy massivlar ustida ishlaydi: bir vaqtda N ta mustaqil hisob.\n"""\nimport numpy as np\n\n# B ilova (majburiy): 1x256 chiziqli zichlash massivi (satr — yuqori yarim bayt)\nZICHLASH = bytes.fromhex(\n    "050d060b010a0f0800040709020c030e"\n    "0807020e0f030b06010c0d0a05040900"\n    "0e020d040c07010b06090005030a080f"\n    "000e090c030d07040f0605010b020a08"\n    "030a0702040c09010e0d0f0800050b06"\n    "02030108000e05090c0b06070a0f0d04"\n    "0a040e0f090508020b0001030c06070d"\n    "0b090a0106040d0f03050e000807020c"\n    "010003070d0b0a0c090e04060f080502"\n    "04080b090e0602050a030c0f070d0001"\n    "090c0f0002010e0a05080b0d04030607"\n    "060b080d07090003040f0a020e010c05"\n    "0f0100050a0803070d02090c060e040b"\n    "0c05040a0b02060d0807030e01000f09"\n    "070f0c060500040e020a080b0d090103"\n    "0d060503080f0c0007010204090b0e0a")\nZ = np.array(list(ZICHLASH), dtype=np.int64)\ndef aes_sbox():\n    s=[0]*256; p=q=1\n    while True:\n        p ^= (p<<1) ^ (0x1b if p&0x80 else 0); p&=0xff\n        q ^= q<<1; q ^= q<<2; q ^= q<<4; q&=0xff\n        if q&0x80: q^=0x09\n        x=q ^ ((q<<1|q>>7)&0xff) ^ ((q<<2|q>>6)&0xff) ^ ((q<<3|q>>5)&0xff) ^ ((q<<4|q>>4)&0xff)\n        s[p]=x^0x63\n        if p==1: break\n    s[0]=0x63; return np.array(s,dtype=np.int64)\nS=aes_sbox()\nclass Cfg:\n    def __init__(s,p=256,bz=\'sbz\'):\n        s.p=p; s.bz=bz\n        s.INV=np.zeros(p,dtype=np.int64)\n        for x in range(1,p,2): s.INV[x]=pow(x,-1,p)\ndef prododd(a,p):\n    b=np.where(a%2==0,a+1,a)%p; r=np.ones(a.shape[0],dtype=np.int64)\n    for i in range(a.shape[1]): r=(r*b[:,i])%p\n    return r\ndef dK(P,Jo,p): return np.where((P==(Jo+1)%p)|(P==(-(Jo+1))%p),10,0)\ndef step(x,A,B,R,c):\n    d=(B+R*x)%c.p; return (-A*x*c.INV[d])%c.p\ndef qosh(h,hn,R,c):\n    p=c.p; R=R[:,None]\n    J1h=h.sum(1)%p; J1hn=hn.sum(1)%p\n    Ph=prododd(h,p); Phn=prododd(hn,p)\n    K1h=(Ph+dK(Ph,J1hn,p))%p; K1hn=(Phn+dK(Phn,J1h,p))%p\n    dAh=np.where(J1hn%2==1,0,1); dAhn=np.where(J1h%2==1,0,1)\n    A1h=(J1hn[:,None]+2*hn+dAh[:,None])%p; A1hn=(J1h[:,None]+2*h+dAhn[:,None])%p\n    h1=step(h,A1h,(K1h[:,None]-2*h)%p,R,c); hn1=step(hn,A1hn,(K1hn[:,None]-2*hn)%p,R,c)\n    J2h=h1.sum(1)%p; J2hn=hn1.sum(1)%p\n    P2h=prododd(h1,p); P2hn=prododd(hn1,p)\n    K2h=(P2h+dK(P2h,J2hn,p))%p; K2hn=(P2hn+dK(P2hn,J2h,p))%p\n    dA2h=np.where((J1hn+J2hn)%2==1,0,1); dA2hn=np.where((J1h+J2h)%2==1,0,1)\n    A2h=(J1hn[:,None]+J2hn[:,None]+2*hn1+dA2h[:,None])%p; A2h=np.where(A2h==0,1,A2h)\n    A2hn=(J1h[:,None]+J2h[:,None]+2*h1+dA2hn[:,None])%p; A2hn=np.where(A2hn==0,1,A2hn)\n    B2h=((K1h*K2h)[:,None]-2*h1)%p; B2hn=((K1hn*K2hn)[:,None]-2*hn1)%p\n    h2=step(h1,A2h,B2h,R,c); h2=np.where(h2==0,(A2h+B2h)%p,h2)\n    hn2=step(hn1,A2hn,B2hn,R,c); hn2=np.where(hn2==0,(A2hn+B2hn)%p,hn2)\n    return step(h2,A2h,B2h,R,c), step(hn2,A2hn,B2hn,R,c)\ndef bzf(h,hn,c):\n    if c.p==256:\n        if c.bz==\'std\': return (Z[h]<<4)|Z[hn]\n        return S[h^S[hn]]\n    if c.bz==\'std\': return h^hn\n    H=16*h[:,0::2]+h[:,1::2]; N=16*hn[:,0::2]+hn[:,1::2]; Q=S[H^S[N]]\n    out=np.empty_like(h); out[:,0::2]=Q>>4; out[:,1::2]=Q&15; return out\ndef dia(H,K,p):  # H,K: (N,4,4)\n    sk=K.sum(1)            # sum_i K[i,u]  -> (N,4)\n    sh=H.sum(1)            # sum_i H[i,u]\n    R=np.empty_like(H)\n    for s in range(4):\n        for u in range(4):\n            if s==u:\n                v=H[:,u,u]*sk[:,u]-sum(H[:,i,i]*K[:,i,u] for i in range(4) if i!=u)\n            else:\n                v=H[:,s,u]*sk[:,u]+K[:,s,u]*sh[:,u]-sum(H[:,s,i]*K[:,i,u] for i in range(4) if i not in (s,u))\n            R[:,s,u]=v%p\n    return R\ndef aralash(h,ke,c):\n    H=h.reshape(-1,4,8); K=ke.reshape(-1,4,8)\n    L=dia(H[:,:,:4],K[:,:,:4],c.p); Rr=dia(H[:,:,4:],K[:,:,4:],c.p)\n    return np.concatenate([L,Rr],axis=2).reshape(-1,32)\ndef _shift(h,rows,cols):\n    m=h.reshape(-1,4,8).copy()\n    for s in range(4): m[:,s,:]=np.roll(m[:,s,:],rows[s],axis=1)\n    for u,n in cols.items(): m[:,:,u]=np.roll(m[:,:,u],n,axis=1)\n    return m.reshape(-1,32)\ndef surholat(h): return _shift(h,[1,2,3,4],{0:1,4:1,1:2,5:2,2:3,6:3})\ndef surkalit(ke): return _shift(ke,[4,3,2,1],{1:-3,5:-3,2:-2,6:-2,3:-1,7:-1})\ndef tuz(ke,k,c):\n    p=c.p\n    return np.where((ke==0)&(k%2==0),2,np.where((ke!=0)&(ke%2==k%2),ke,(ke-1)%p))\ndef iteration(holat,blok,ke,k,c,trace=None,rounds=10):\n    """Bitta tashqi iteratsiya (i): Qo\'sh+BZ, so\'ng \'rounds\' ta bosqich."""\n    R=k[:,0]\n    def T(name,j,h,kk):\n        if trace is not None: trace.append((name,j,h.copy(),kk.copy()))\n    holat,hn=qosh(holat,blok,R,c); T(\'Qosh0\',0,holat,ke)\n    holat=bzf(holat,hn,c); T(\'BZ0\',0,holat,ke)\n    for j in range(1,rounds+1):\n        holat=aralash(holat,ke,c); T(\'Aralash\',j,holat,ke)\n        holat,ke=qosh(holat,ke,R,c); T(\'Qosh1\',j,holat,ke)\n        holat=surholat(holat); ke=surkalit(ke); T(\'Sur\',j,holat,ke)\n        if j==8:\n            holat,hn=qosh(holat,blok,R,c); T(\'Qosh_inj8\',j,holat,ke); holat=bzf(holat,hn,c); T(\'BZ8\',j,holat,ke)\n        else:\n            holat,ke=qosh(holat,ke,R,c); T(\'Qosh2\',j,holat,ke)\n        ke=tuz(ke,k,c); T(\'Tuz\',j,holat,ke)\n        if j==9:\n            holat,hn=qosh(holat,blok,R,c); T(\'Qosh_inj9\',j,holat,ke); holat=bzf(holat,hn,c); T(\'BZ9\',j,holat,ke)\n    return holat,ke\ndef to_units(b,p):\n    b=np.asarray(b,dtype=np.int64)\n    if p==256: return b\n    out=np.empty(b.shape[:-1]+(b.shape[-1]*2,),dtype=np.int64); out[...,0::2]=b>>4; out[...,1::2]=b&15; return out\ndef hash_msgs(msgs,keys,c):\n    """msgs: (N,L) bayt, L - blok uzunligiga karrali; keys: (N,32 yoki 16)."""\n    return _hash_core(msgs,keys,c,bits=msgs.shape[1]*8)\ndef _hash_core(msgs,keys,c,bits):\n    N,L=msgs.shape; bl=32 if c.p==256 else 16\n    nb=L//bl\n    blocks=[msgs[:,i*bl:(i+1)*bl] for i in range(nb)]\n    ny=[0]*N\n    nyb=np.zeros((N,bl),dtype=np.int64)\n    for n in range(N):\n        v=sum(int.from_bytes(bytes(b[n].tolist()),\'big\') for b in blocks)%(2**(8*bl))\n        nyb[n]=list(v.to_bytes(bl,\'big\'))\n    lb=np.tile(np.array(list(bits.to_bytes(bl,\'big\'))),(N,1))\n    kir=[to_units(b,c.p) for b in blocks]+[to_units(lb,c.p),to_units(nyb,c.p)]\n    k=to_units(keys,c.p); ke=k.copy(); holat=kir[0]\n    for i in range(1,len(kir)): holat,ke=iteration(holat,kir[i],ke,k,c)\n    return holat\n\n\n# ------------------------------------------------------------------ nazorat qiymatlari\nKAT_KALIT = bytes.fromhex("c86ed4b2756df1cd6b6eca81d441809221ab81bba0c7c15044be8feb934f95a2")\nKAT_XABAR = b"funksiyaxeshirovaniyainformasionnayatexnologiyakriptograficheska"\nKAT = {\n    (256, \'std\'): "55394f2d84fccd4d8d01d8fef756cd657ebecc3f96f81e5207767447de66e2b6",\n    (256, \'sbz\'): "42f97daaa8e7010bc629ca695eaa77df7ae7443a0c3ae78d205c2e025d256c96",\n    (16, \'std\'):  "e9eaccdd4ccce2d229228c864989eb9f",\n    (16, \'sbz\'):  "bd745e2ed572ea4246ed68482ef46c5e",\n}\n\ndef xesh(xabar: bytes, kalit: bytes, p=256, bz=\'sbz\') -> bytes:\n    """Bitta xabar uchun xesh qiymat (xabar uzunligi blok uzunligiga karrali bo‘lishi shart emas)."""\n    bl = 32 if p == 256 else 16\n    m = xabar + b"\\x00" * (-len(xabar) % bl) if xabar else b"\\x00" * bl\n    # uzunlik bloki asl xabar uzunligi bo‘yicha hisoblanadi\n    c = Cfg(p, bz)\n    msgs = np.array([list(m)]); keys = np.array([list(kalit[:bl])])\n    out = _hash_core(msgs, keys, c, bits=len(xabar) * 8)[0]\n    if p == 16: out = [out[2*i] << 4 | out[2*i+1] for i in range(16)]\n    return bytes(int(x) for x in out)\n\ndef nazorat():\n    """A ilova va SBZ materialidagi nazorat qiymatlarini tekshiradi."""\n    ok = True\n    for (p, bz), kut in KAT.items():\n        r = xesh(KAT_XABAR, KAT_KALIT, p, bz).hex()\n        print(f"  p={p:3d}, {bz}: {\'MOS\' if r == kut else \'MOS EMAS\'}")\n        ok &= (r == kut)\n    return ok\n\nif __name__ == "__main__":\n    print("Nazorat qiymatlari:"); print("Hammasi mos" if nazorat() else "XATO")\n',
    'komponentlar': '"""Kriptotahlil uchun umumiy yordamchi funksiyalar."""\nimport itertools\nimport numpy as np\nfrom algoritm import Cfg, aralash, surholat\n\nPAR8 = np.array([bin(i).count(\'1\') & 1 for i in range(256)])\nX256 = np.arange(256)\n\ndef kalitlar(rng, N, p):\n    """Tasodifiy kalitlar: R (1-element) va 2-element juft (5.1.2-band talabi)."""\n    if p == 256:\n        k = rng.integers(0, 256, (N, 32)); k[:, 0] &= 0xFE; k[:, 1] &= 0xFE\n    else:\n        k = rng.integers(0, 16, (N, 32)); k[:, 0] &= 0xE; k[:, 1] &= 0xE\n    return k\n\ndef wht(f):\n    """Walsh–Adamar o‘zgartirishi (f — ±1 vektor)."""\n    a = f.astype(np.int64).copy(); h = 1; n = len(a)\n    while h < n:\n        a = a.reshape(-1, 2, h); a = np.stack([a[:, 0] + a[:, 1], a[:, 0] - a[:, 1]], 1).reshape(-1); h *= 2\n    return a\n\ndef lat(F, nout):\n    """8 bitli kirishli F uchun W[β][α] = Σ (−1)^(β·F(x) ⊕ α·x); korrelyatsiya c = W/256."""\n    W = np.zeros((1 << nout, 256), dtype=np.int64)\n    for b in range(1 << nout):\n        W[b] = wht(1 - 2 * PAR8[F & b])\n    return W\n\ndef ddt(F, nout):\n    """Ayirmalar matritsasi DDT[Δx][Δy] (8 bitli kirish)."""\n    return np.array([np.bincount(F ^ F[X256 ^ d], minlength=nout) for d in range(256)])\n\ndef anf_deg(fbits, nv):\n    """Boolean funksiyaning ANF darajasi (Möbius o‘zgartirishi)."""\n    a = fbits.astype(np.uint8).copy(); h = 1; N = 1 << nv\n    while h < N:\n        a = a.reshape(-1, 2, h); a[:, 1] ^= a[:, 0]; a = a.reshape(-1); h *= 2\n    wt = np.array([bin(i).count(\'1\') for i in range(N)]); nz = wt[a == 1]\n    return int(nz.max()) if len(nz) else -1\n\ndef bits(v, n): return (v[:, None] >> np.arange(n)) & 1\n\ndef implicit_count(points, deg):\n    """deg darajali chiziqli bog‘liq bo‘lmagan implitsit tenglamalar soni (monomlar matritsasi rangi orqali)."""\n    npts, nv = points.shape\n    mons = [()] + [m for d in range(1, deg + 1) for m in itertools.combinations(range(nv), d)]\n    words = (npts + 63) // 64; cols = []\n    for m in mons:\n        v = np.ones(npts, dtype=np.uint8)\n        for i in m: v &= points[:, i].astype(np.uint8)\n        pad = np.zeros(words * 64, dtype=np.uint8); pad[:npts] = v\n        cols.append(np.packbits(pad).view(np.uint64))\n    A = np.array(cols); rank = 0; n = A.shape[0]\n    for w in range(A.shape[1]):\n        for b in range(64):\n            if rank >= n: break\n            mask = np.uint64(1) << np.uint64(b)\n            idx = np.nonzero((A[rank:, w] & mask) != 0)[0]\n            if len(idx) == 0: continue\n            piv = rank + idx[0]\n            if piv != rank: A[[rank, piv]] = A[[piv, rank]]\n            rows = np.nonzero((A[:, w] & mask) != 0)[0]; rows = rows[rows != rank]\n            A[rows] ^= A[rank]; rank += 1\n    return len(mons) - rank, len(mons)\n\ndef qosh_elem(A1, B1, A2, B2, R, p=256):\n    """Qo‘sh o‘zgartirishining bitta elementi (parametrlar belgilangan): x → x^(−3)."""\n    x = np.arange(p); INV = np.zeros(p, dtype=np.int64)\n    for t in range(1, p, 2): INV[t] = pow(t, -1, p)\n    s = lambda x, A, B: (-A * x * INV[(B + R * x) % p]) % p\n    y1 = s(x, A1, B1); y2 = s(y1, A2, B2); y2 = np.where(y2 == 0, (A2 + B2) % p, y2)\n    return s(y2, A2, B2)\n\ndef tasodifiy_param(rng):\n    A1, B1, A2, B2 = [int(rng.integers(0, 128)) * 2 + 1 for _ in range(4)]\n    return A1, B1, A2, B2, int(rng.integers(0, 128)) * 2\n\ndef rank2(A):\n    A = A.copy() % 2; r = 0\n    for col in range(A.shape[1]):\n        piv = [i for i in range(r, A.shape[0]) if A[i, col]]\n        if not piv: continue\n        A[[r, piv[0]]] = A[[piv[0], r]]\n        for i in range(A.shape[0]):\n            if i != r and A[i, col]: A[i] ^= A[r]\n        r += 1\n    return r\n\n# 0-qatlam matritsalari: SurHolat (P) va Aralash (M(k0))\n_C2 = Cfg(2, \'std\')\n_perm = surholat(np.arange(32)[None, :].copy())[0]\nP_SUR = np.zeros((32, 32), dtype=np.int64); P_SUR[np.arange(32), _perm] = 1\n\ndef M_aralash(k):\n    """h0\' = M(k0)·h0 (mod 2) matritsasi."""\n    E = np.eye(32, dtype=np.int64); K = np.tile(np.asarray(k) % 2, (32, 1))\n    return (aralash(E, K, _C2).T) % 2\n\ndef L_raund(k):\n    """Yutishsiz raundning 0-qatlam matritsasi L = P·M(k0)."""\n    return (P_SUR @ M_aralash(k)) % 2\n',
    'variant': '"""\nTavsiyalar bo‘yicha o‘zgartirilgan 1-algoritm variantlari (vektorlashtirilgan).\n  bz    : \'std\' (BaytZichlash) yoki \'sbz\' (S(h ⊕ S(n)))      — 1-tavsiya\n  toq   : True — kₑ elementlari doim toq (kₑ ← kₑ ∨ 1)         — 2-tavsiya\n  sqatlam: True — raund ichida S-box qatlami (SurHolat dan keyin) — 3-tavsiya\n"""\nimport numpy as np\nfrom algoritm import Cfg, aralash, qosh, surholat, surkalit, tuz, bzf, S, to_units\n\nPRESENT = np.array([0xC, 5, 6, 0xB, 9, 0, 0xA, 0xD, 3, 0xE, 0xF, 8, 4, 7, 1, 2])\n\ndef s_qatlam(h, p):\n    return S[h] if p == 256 else PRESENT[h]\n\ndef iteration_v(holat, blok, ke, k, c, toq=False, sqatlam=False, trace=None, rounds=10):\n    R = k[:, 0]\n    def T(nm, j):\n        if trace is not None: trace.append((nm, j, holat.copy(), ke.copy()))\n    if toq: ke = ke | 1\n    holat, hn = qosh(holat, blok, R, c); T(\'Qosh0\', 0)\n    holat = bzf(holat, hn, c); T(\'BZ0\', 0)\n    for j in range(1, rounds + 1):\n        holat = aralash(holat, ke, c); T(\'Aralash\', j)\n        holat, ke = qosh(holat, ke, R, c); T(\'Qosh1\', j)\n        holat = surholat(holat); ke = surkalit(ke); T(\'Sur\', j)\n        if sqatlam: holat = s_qatlam(holat, c.p); T(\'Sqatlam\', j)\n        if j == 8:\n            holat, hn = qosh(holat, blok, R, c); holat = bzf(holat, hn, c); T(\'BZ8\', j)\n        else:\n            holat, ke = qosh(holat, ke, R, c); T(\'Qosh2\', j)\n        ke = (ke | 1) if toq else tuz(ke, k, c); T(\'Tuz\', j)\n        if j == 9:\n            holat, hn = qosh(holat, blok, R, c); holat = bzf(holat, hn, c); T(\'BZ9\', j)\n    return holat, ke\n\ndef xesh_v(xabar: bytes, kalit: bytes, p=256, bz=\'sbz\', toq=False, sqatlam=False) -> bytes:\n    c = Cfg(p, bz); bl = 32 if p == 256 else 16\n    bits = len(xabar) * 8\n    m = xabar + b"\\x00" * (-len(xabar) % bl) if xabar else b"\\x00" * bl\n    bloks = [np.array(list(m[i:i + bl])) for i in range(0, len(m), bl)]\n    ny = sum(int.from_bytes(bytes(b.tolist()), \'big\') for b in bloks) % 2 ** (8 * bl)\n    kir = [to_units(b, p) for b in bloks] + [to_units(np.array(list(bits.to_bytes(bl, \'big\'))), p),\n                                             to_units(np.array(list(ny.to_bytes(bl, \'big\'))), p)]\n    k = to_units(np.array(list(kalit[:bl])), p)[None]; ke = k.copy(); h = kir[0][None]\n    for i in range(1, len(kir)):\n        h, ke = iteration_v(h, kir[i][None], ke, k, c, toq, sqatlam)\n    h = h[0]\n    if p == 16: h = [h[2 * i] << 4 | h[2 * i + 1] for i in range(16)]\n    return bytes(int(v) for v in h)\n',
}
for _nom, _matn in _MODULLAR.items():
    if importlib.util.find_spec(_nom) is not None and _nom not in sys.modules:
        print(f"{_nom}.py — papkadagi fayldan olinadi"); continue
    _m = types.ModuleType(_nom); _m.__file__ = f"<{_nom}>"
    sys.modules[_nom] = _m; exec(compile(_matn, f"<{_nom}>", "exec"), _m.__dict__)
    print(f"{_nom} — daftar ichidan yuklandi")


algoritm — daftar ichidan yuklandi
komponentlar — daftar ichidan yuklandi
variant — daftar ichidan yuklandi


In [2]:
import numpy as np, json, os, sys, time, shutil, subprocess, statistics
from algoritm import *
from komponentlar import kalitlar
from variant import xesh_v, iteration_v
VAR = [('std', 'std', 0, 0, 0), ('std+toq', 'std', 0, 1, 0), ('std+S', 'std', 0, 0, 1),
       ('sbz', 'sbz', 1, 0, 0), ('sbz+toq', 'sbz', 1, 1, 0), ('sbz+toq+S', 'sbz', 1, 1, 1)]
XABAR_HAJMI = 262144      # C o‘lchovi uchun xabar uzunligi (bayt)
TAKROR = 5                # har bir o‘lchov takrori (eng yaxshi natija olinadi)
TEZ = False
if TEZ: XABAR_HAJMI, TAKROR = 65536, 2
NATIJA = {}
print("Python: nazorat qiymatlari")
PYKAT = {}
for p in (256, 16):
    for nom, bz, _, toq, sq in VAR:
        PYKAT[(p, nom)] = xesh_v(KAT_XABAR, KAT_KALIT, p, bz, bool(toq), bool(sq)).hex()
        print(f"  p={p:3d}  {nom:10s} {PYKAT[(p, nom)]}")

Python: nazorat qiymatlari
  p=256  std        55394f2d84fccd4d8d01d8fef756cd657ebecc3f96f81e5207767447de66e2b6
  p=256  std+toq    073956b2e84a40721e49415fca606b118fc65c174d511fda3fde998f75a024fb
  p=256  std+S      9f9c061f419d14358fe7c935dc0afc77561dc61e0e2d23c4aa32c6a7ff044e7c
  p=256  sbz        42f97daaa8e7010bc629ca695eaa77df7ae7443a0c3ae78d205c2e025d256c96
  p=256  sbz+toq    2e632c6921f4ecf801e1a5ab1d056a5b1425ee6117a17d4ec8b3e982d66b10b7
  p=256  sbz+toq+S  240aa5e0f429d55372144ea1091c7c50caea0c623c659b34cf82f1e04962c4ba
  p= 16  std        e9eaccdd4ccce2d229228c864989eb9f
  p= 16  std+toq    c548c7a15798b147b51ea774d174b6c1
  p= 16  std+S      9bf627ae97537cd7a99f68dd15d169c4
  p= 16  sbz        bd745e2ed572ea4246ed68482ef46c5e
  p= 16  sbz+toq    e4d3acb3d969ce8cfeaf56f521c4c17f
  p= 16  sbz+toq+S  447fe9b4f426a47dda51492373cbf610


## 1. C realizatsiyasi

Quyidagi hujayra C manba kodini `uzmst285.c` fayliga yozadi, uni kompilyatsiya qiladi va nazorat qiymatlarini
Python realizatsiyasi bilan solishtiradi. C dastur argumentlari: `kat` — nazorat qiymatlari;
`p bz toq sq bayt_soni takror` — tezlik o‘lchovi (sekundlarda eng yaxshi vaqt).

In [3]:
C_MANBA = '/* O\'zMSt 285:2024, 1-algoritm: C realizatsiyasi (tezlikni o\'lchash uchun).\n   Variantlar: p = 256 yoki 16; bz = 0 (BaytZichlash) yoki 1 (SBZ);\n   toq = 1 — barcha kₑ elementlari toq (2-tavsiya); sqatlam = 1 — raund ichidagi S-box qatlami (3-tavsiya). */\n#include <stdint.h>\n#include <string.h>\n#include <stdio.h>\n#include <stdlib.h>\n#include <time.h>\n\nstatic const uint8_t ZICH[256] = {\n 5,13,6,11,1,10,15,8,0,4,7,9,2,12,3,14, 8,7,2,14,15,3,11,6,1,12,13,10,5,4,9,0,\n 14,2,13,4,12,7,1,11,6,9,0,5,3,10,8,15, 0,14,9,12,3,13,7,4,15,6,5,1,11,2,10,8,\n 3,10,7,2,4,12,9,1,14,13,15,8,0,5,11,6, 2,3,1,8,0,14,5,9,12,11,6,7,10,15,13,4,\n 10,4,14,15,9,5,8,2,11,0,1,3,12,6,7,13, 11,9,10,1,6,4,13,15,3,5,14,0,8,7,2,12,\n 1,0,3,7,13,11,10,12,9,14,4,6,15,8,5,2, 4,8,11,9,14,6,2,5,10,3,12,15,7,13,0,1,\n 9,12,15,0,2,1,14,10,5,8,11,13,4,3,6,7, 6,11,8,13,7,9,0,3,4,15,10,2,14,1,12,5,\n 15,1,0,5,10,8,3,7,13,2,9,12,6,14,4,11, 12,5,4,10,11,2,6,13,8,7,3,14,1,0,15,9,\n 7,15,12,6,5,0,4,14,2,10,8,11,13,9,1,3, 13,6,5,3,8,15,12,0,7,1,2,4,9,11,14,10};\nstatic uint8_t SB[256];\nstatic const uint8_t PRESENT[16] = {0xC,5,6,0xB,9,0,0xA,0xD,3,0xE,0xF,8,4,7,1,2};\nstatic uint32_t INV256[256], INV16[16];\n\nstatic void init_tables(void) {\n    uint8_t p = 1, q = 1;\n    do {\n        p = p ^ (uint8_t)(p << 1) ^ ((p & 0x80) ? 0x1B : 0);\n        q ^= q << 1; q ^= q << 2; q ^= q << 4; if (q & 0x80) q ^= 0x09;\n        uint8_t x = q ^ (uint8_t)((q << 1) | (q >> 7)) ^ (uint8_t)((q << 2) | (q >> 6)) ^ (uint8_t)((q << 3) | (q >> 5)) ^ (uint8_t)((q << 4) | (q >> 4));\n        SB[p] = x ^ 0x63;\n    } while (p != 1);\n    SB[0] = 0x63;\n    for (uint32_t x = 1; x < 256; x += 2) for (uint32_t y = 1; y < 256; y += 2) if (((x * y) & 255) == 1) INV256[x] = y;\n    for (uint32_t x = 1; x < 16; x += 2) for (uint32_t y = 1; y < 16; y += 2) if (((x * y) & 15) == 1) INV16[x] = y;\n}\n\ntypedef struct { uint32_t p, m; const uint32_t *inv; int bz, toq, sq; } cfg_t;\n\nstatic inline uint32_t step(uint32_t x, uint32_t A, uint32_t B, uint32_t R, const cfg_t *c) {\n    uint32_t d = (B + R * x) & c->m;\n    return (0u - A * x * c->inv[d]) & c->m;\n}\nstatic inline uint32_t prododd(const uint32_t *a, const cfg_t *c) {\n    uint32_t r = 1; for (int i = 0; i < 32; i++) r = (r * (a[i] | 1)) & c->m; return r;\n}\nstatic inline uint32_t dK(uint32_t P, uint32_t Jo, const cfg_t *c) {\n    uint32_t t = (Jo + 1) & c->m; return (P == t || P == ((0u - t) & c->m)) ? 10 : 0;\n}\nstatic void qosh(uint32_t *h, uint32_t *hn, uint32_t R, const cfg_t *c) {\n    uint32_t m = c->m, J1h = 0, J1hn = 0, h1[32], hn1[32];\n    for (int i = 0; i < 32; i++) { J1h += h[i]; J1hn += hn[i]; }\n    J1h &= m; J1hn &= m;\n    uint32_t Ph = prododd(h, c), Phn = prododd(hn, c);\n    uint32_t K1h = (Ph + dK(Ph, J1hn, c)) & m, K1hn = (Phn + dK(Phn, J1h, c)) & m;\n    uint32_t dAh = (J1hn & 1) ? 0 : 1, dAhn = (J1h & 1) ? 0 : 1;\n    for (int i = 0; i < 32; i++) {\n        h1[i]  = step(h[i],  (J1hn + 2 * hn[i] + dAh) & m, (K1h - 2 * h[i]) & m, R, c);\n        hn1[i] = step(hn[i], (J1h + 2 * h[i] + dAhn) & m, (K1hn - 2 * hn[i]) & m, R, c);\n    }\n    uint32_t J2h = 0, J2hn = 0;\n    for (int i = 0; i < 32; i++) { J2h += h1[i]; J2hn += hn1[i]; }\n    J2h &= m; J2hn &= m;\n    uint32_t P2h = prododd(h1, c), P2hn = prododd(hn1, c);\n    uint32_t K2h = (P2h + dK(P2h, J2hn, c)) & m, K2hn = (P2hn + dK(P2hn, J2h, c)) & m;\n    uint32_t dA2h = ((J1hn + J2hn) & 1) ? 0 : 1, dA2hn = ((J1h + J2h) & 1) ? 0 : 1;\n    uint32_t KKh = (K1h * K2h) & m, KKhn = (K1hn * K2hn) & m;\n    for (int i = 0; i < 32; i++) {\n        uint32_t A2 = (J1hn + J2hn + 2 * hn1[i] + dA2h) & m; if (!A2) A2 = 1;\n        uint32_t B2 = (KKh - 2 * h1[i]) & m;\n        uint32_t x2 = step(h1[i], A2, B2, R, c); if (!x2) x2 = (A2 + B2) & m;\n        uint32_t A2n = (J1h + J2h + 2 * h1[i] + dA2hn) & m; if (!A2n) A2n = 1;\n        uint32_t B2n = (KKhn - 2 * hn1[i]) & m;\n        uint32_t y2 = step(hn1[i], A2n, B2n, R, c); if (!y2) y2 = (A2n + B2n) & m;\n        h[i] = step(x2, A2, B2, R, c); hn[i] = step(y2, A2n, B2n, R, c);\n    }\n}\nstatic void dia(uint32_t *h, const uint32_t *k, int off, const cfg_t *c) {\n    uint32_t H[4][4], K[4][4], sk[4], sh[4], m = c->m;\n    for (int s = 0; s < 4; s++) for (int u = 0; u < 4; u++) { H[s][u] = h[8 * s + off + u]; K[s][u] = k[8 * s + off + u]; }\n    for (int u = 0; u < 4; u++) { sk[u] = K[0][u] + K[1][u] + K[2][u] + K[3][u]; sh[u] = H[0][u] + H[1][u] + H[2][u] + H[3][u]; }\n    for (int s = 0; s < 4; s++) for (int u = 0; u < 4; u++) {\n        uint32_t v;\n        if (s == u) { v = H[u][u] * sk[u]; for (int i = 0; i < 4; i++) if (i != u) v -= H[i][i] * K[i][u]; }\n        else { v = H[s][u] * sk[u] + K[s][u] * sh[u]; for (int i = 0; i < 4; i++) if (i != s && i != u) v -= H[s][i] * K[i][u]; }\n        h[8 * s + off + u] = v & m;\n    }\n}\nstatic void aralash(uint32_t *h, const uint32_t *ke, const cfg_t *c) { dia(h, ke, 0, c); dia(h, ke, 4, c); }\nstatic void shift(uint32_t *a, const int *rows, const int *cols) {\n    uint32_t t[32];\n    for (int s = 0; s < 4; s++) for (int u = 0; u < 8; u++) t[8 * s + u] = a[8 * s + ((u - rows[s]) % 8 + 8) % 8];\n    for (int u = 0; u < 8; u++) for (int s = 0; s < 4; s++) a[8 * s + u] = t[8 * (((s - cols[u]) % 4 + 4) % 4) + u];\n}\nstatic const int HR[4] = {1, 2, 3, 4}, HC[8] = {1, 2, 3, 0, 1, 2, 3, 0};\nstatic const int KR[4] = {4, 3, 2, 1}, KC[8] = {0, -3, -2, -1, 0, -3, -2, -1};\nstatic void tuz(uint32_t *ke, const uint32_t *k, const cfg_t *c) {\n    for (int i = 0; i < 32; i++) {\n        if (c->toq) { ke[i] |= 1; continue; }\n        if (ke[i] == 0 && !(k[i] & 1)) ke[i] = 2;\n        else if (ke[i] != 0 && (ke[i] & 1) == (k[i] & 1)) ;\n        else ke[i] = (ke[i] - 1) & c->m;\n    }\n}\nstatic void bzf(uint32_t *h, const uint32_t *hn, const cfg_t *c) {\n    if (c->p == 256) {\n        for (int i = 0; i < 32; i++) h[i] = c->bz ? SB[h[i] ^ SB[hn[i]]] : (uint32_t)((ZICH[h[i]] << 4) | ZICH[hn[i]]);\n    } else {\n        if (!c->bz) { for (int i = 0; i < 32; i++) h[i] ^= hn[i]; return; }\n        for (int j = 0; j < 16; j++) {\n            uint32_t H = 16 * h[2 * j] + h[2 * j + 1], N = 16 * hn[2 * j] + hn[2 * j + 1], Q = SB[H ^ SB[N]];\n            h[2 * j] = Q >> 4; h[2 * j + 1] = Q & 15;\n        }\n    }\n}\nstatic void sqatlam(uint32_t *h, const cfg_t *c) {\n    if (!c->sq) return;\n    for (int i = 0; i < 32; i++) h[i] = (c->p == 256) ? SB[h[i]] : PRESENT[h[i]];\n}\nstatic void iteration(uint32_t *holat, const uint32_t *blok, uint32_t *ke, const uint32_t *k, const cfg_t *c) {\n    uint32_t R = k[0], hn[32];\n    if (c->toq) for (int i = 0; i < 32; i++) ke[i] |= 1;\n    memcpy(hn, blok, sizeof hn); qosh(holat, hn, R, c); bzf(holat, hn, c);\n    for (int j = 1; j <= 10; j++) {\n        aralash(holat, ke, c);\n        qosh(holat, ke, R, c);\n        shift(holat, HR, HC); shift(ke, KR, KC);\n        sqatlam(holat, c);\n        if (j == 8) { memcpy(hn, blok, sizeof hn); qosh(holat, hn, R, c); bzf(holat, hn, c); }\n        else qosh(holat, ke, R, c);\n        tuz(ke, k, c);\n        if (j == 9) { memcpy(hn, blok, sizeof hn); qosh(holat, hn, R, c); bzf(holat, hn, c); }\n    }\n}\nstatic void units(const uint8_t *b, int bl, uint32_t *u, int p) {\n    if (p == 256) for (int i = 0; i < 32; i++) u[i] = b[i];\n    else for (int i = 0; i < bl; i++) { u[2 * i] = b[i] >> 4; u[2 * i + 1] = b[i] & 15; }\n}\n/* xesh: xabar, uzunlik (bayt), kalit (32 yoki 16 bayt) -> chiqish (32 yoki 16 bayt) */\nvoid xesh(const uint8_t *msg, size_t len, const uint8_t *key, int p, int bz, int toq, int sq, uint8_t *out) {\n    cfg_t c = {(uint32_t)p, (uint32_t)p - 1, p == 256 ? INV256 : INV16, bz, toq, sq};\n    int bl = p == 256 ? 32 : 16;\n    size_t nb = len ? (len + bl - 1) / bl : 1;\n    uint8_t blk[32], ny[32] = {0}, lb[32] = {0};\n    uint32_t holat[32], blok[32], ke[32], k[32];\n    uint64_t bits = (uint64_t)len * 8;\n    for (int i = 0; i < 8; i++) lb[bl - 1 - i] = (uint8_t)(bits >> (8 * i));\n    units(key, bl, k, p); memcpy(ke, k, sizeof ke);\n    for (size_t b = 0; b < nb + 2; b++) {\n        if (b < nb) {\n            memset(blk, 0, sizeof blk);\n            size_t n = (b + 1) * bl <= len ? (size_t)bl : (len > b * bl ? len - b * bl : 0);\n            if (n) memcpy(blk, msg + b * bl, n);\n            unsigned carry = 0;\n            for (int i = bl - 1; i >= 0; i--) { unsigned s = ny[i] + blk[i] + carry; ny[i] = s & 255; carry = s >> 8; }\n        } else memcpy(blk, b == nb ? lb : ny, bl);\n        if (b == 0) { units(blk, bl, holat, p); continue; }\n        units(blk, bl, blok, p); iteration(holat, blok, ke, k, &c);\n    }\n    if (p == 256) for (int i = 0; i < 32; i++) out[i] = (uint8_t)holat[i];\n    else for (int i = 0; i < 16; i++) out[i] = (uint8_t)(holat[2 * i] << 4 | holat[2 * i + 1]);\n}\nstatic double now(void) { return (double)clock() / CLOCKS_PER_SEC; }\nint main(int argc, char **argv) {\n    init_tables();\n    static const uint8_t KEY[32] = {0xc8,0x6e,0xd4,0xb2,0x75,0x6d,0xf1,0xcd,0x6b,0x6e,0xca,0x81,0xd4,0x41,0x80,0x92,\n                                    0x21,0xab,0x81,0xbb,0xa0,0xc7,0xc1,0x50,0x44,0xbe,0x8f,0xeb,0x93,0x4f,0x95,0xa2};\n    const char *M = "funksiyaxeshirovaniyainformasionnayatexnologiyakriptograficheska";\n    uint8_t out[32];\n    if (argc > 1 && !strcmp(argv[1], "kat")) {               /* nazorat qiymatlari */\n        for (int p = 256; p >= 16; p /= 16) for (int v = 0; v < 6; v++) {\n            int bz = v >= 1, toq = v >= 2, sq = v >= 3;\n            if (v == 4) { bz = 0; toq = 1; sq = 0; } if (v == 5) { bz = 0; toq = 0; sq = 1; }\n            xesh((const uint8_t *)M, 64, KEY, p, bz, toq, sq, out);\n            printf("%d %d %d %d ", p, bz, toq, sq); for (int i = 0; i < (p == 256 ? 32 : 16); i++) printf("%02x", out[i]); printf("\\n");\n        }\n        return 0;\n    }\n    /* tezlik: argv = p bz toq sq bayt_soni takror */\n    int p = atoi(argv[1]), bz = atoi(argv[2]), toq = atoi(argv[3]), sq = atoi(argv[4]);\n    size_t n = (size_t)atol(argv[5]); int rep = atoi(argv[6]);\n    uint8_t *buf = malloc(n); for (size_t i = 0; i < n; i++) buf[i] = (uint8_t)(i * 131 + 7);\n    double best = 1e9;\n    for (int r = 0; r < rep; r++) { double t = now(); xesh(buf, n, KEY, p, bz, toq, sq, out); t = now() - t; if (t < best) best = t; }\n    printf("%.6f %02x\\n", best, out[0]);\n    free(buf); return 0;\n}\n'

In [4]:
open('uzmst285.c', 'w', encoding='utf-8').write(C_MANBA)
EXE = os.path.abspath('uzmst285.exe' if sys.platform.startswith('win') else 'uzmst285')
KOMPILYATOR = next((c for c in ('gcc', 'cc', 'clang') if shutil.which(c)), None)
C_BOR = False
if KOMPILYATOR:
    r = subprocess.run([KOMPILYATOR, '-O2', '-o', EXE, 'uzmst285.c'], capture_output=True, text=True)
    C_BOR = (r.returncode == 0)
    print("Kompilyator:", KOMPILYATOR, "— muvaffaqiyatli" if C_BOR else "— xato:\n" + r.stderr)
    if C_BOR: print(subprocess.run([KOMPILYATOR, '--version'], capture_output=True, text=True).stdout.splitlines()[0])
else:
    print("C kompilyatori topilmadi: C o‘lchovi o‘tkazib yuboriladi (Python va model natijalari baribir hisoblanadi).")
if C_BOR:
    nom_kod = {(bz, toq, sq): nom for nom, _, bz, toq, sq in VAR}
    ok = True
    for line in subprocess.run([EXE, 'kat'], capture_output=True, text=True).stdout.split('\n'):
        if not line.strip(): continue
        p, bz, toq, sq, h = line.split(); key = (int(p), nom_kod[(int(bz), int(toq), int(sq))])
        mos = (PYKAT[key] == h); ok &= mos
        print(f"  p={p:>3s}  {key[1]:10s} C = Python: {'ha' if mos else 'YO‘Q'}")
    print("Barcha variantlar bo‘yicha C va Python natijalari mos:", ok)

Kompilyator: gcc — muvaffaqiyatli
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
  p=256  std        C = Python: ha
  p=256  sbz        C = Python: ha
  p=256  sbz+toq    C = Python: ha
  p=256  sbz+toq+S  C = Python: ha
  p=256  std+toq    C = Python: ha
  p=256  std+S      C = Python: ha
  p= 16  std        C = Python: ha
  p= 16  sbz        C = Python: ha
  p= 16  sbz+toq    C = Python: ha
  p= 16  sbz+toq+S  C = Python: ha
  p= 16  std+toq    C = Python: ha
  p= 16  std+S      C = Python: ha
Barcha variantlar bo‘yicha C va Python natijalari mos: True


In [5]:
if C_BOR:
    NATIJA['C'] = {}
    o_lchov = {}
    for tur in range(3):                         # variantlar aralash tartibda 3 marta o‘lchanadi
        for p in (256, 16):
            for nom, _, bz, toq, sq in VAR:
                out = subprocess.run([EXE, str(p), str(bz), str(toq), str(sq), str(XABAR_HAJMI), str(TAKROR)], capture_output=True, text=True).stdout.split()
                o_lchov.setdefault((p, nom), []).append(float(out[0]))
    print(f"Xabar uzunligi {XABAR_HAJMI} bayt; har bir variant uchun eng yaxshi vaqt:")
    print(f"{'rejim':>6s} {'variant':>11s} {'vaqt, ms':>9s} {'mkS/blok':>9s} {'MB/s':>7s} {'farq, %':>8s}")
    for p in (256, 16):
        bl = 32 if p == 256 else 16; nb = XABAR_HAJMI // bl + 2; asos = min(o_lchov[(p, 'std')])
        for nom, *_ in VAR:
            t = min(o_lchov[(p, nom)])
            NATIJA['C'][f'{p}_{nom}'] = dict(vaqt_ms=round(t * 1e3, 2), mks_blok=round(t / nb * 1e6, 3), MBs=round(XABAR_HAJMI / t / 1e6, 2), farq_foiz=round((t / asos - 1) * 100, 1))
            d = NATIJA['C'][f'{p}_{nom}']
            print(f"{p:6d} {nom:>11s} {d['vaqt_ms']:9.2f} {d['mks_blok']:9.3f} {d['MBs']:7.2f} {d['farq_foiz']:+8.1f}")
    # o‘lchov shovqini: bir xil variantning takroriy o‘lchovlari orasidagi farq
    sh = max((max(v) / min(v) - 1) * 100 for v in o_lchov.values())
    NATIJA['C_shovqin_foiz'] = round(sh, 1); print(f"Takroriy o‘lchovlar orasidagi eng katta farq (shovqin): {sh:.1f}%")

Xabar uzunligi 262144 bayt; har bir variant uchun eng yaxshi vaqt:
 rejim     variant  vaqt, ms  mkS/blok    MB/s  farq, %
   256         std     58.52     7.142    4.48     +0.0
   256     std+toq     57.98     7.076    4.52     -0.9
   256       std+S     59.60     7.274    4.40     +1.9
   256         sbz     58.97     7.197    4.45     +0.8
   256     sbz+toq     58.25     7.108    4.50     -0.5
   256   sbz+toq+S     59.14     7.217    4.43     +1.1
    16         std    117.83     7.191    2.22     +0.0
    16     std+toq    119.14     7.271    2.20     +1.1
    16       std+S    119.41     7.288    2.20     +1.3
    16         sbz    117.26     7.156    2.24     -0.5
    16     sbz+toq    118.46     7.230    2.21     +0.5
    16   sbz+toq+S    120.03     7.325    2.18     +1.9
Takroriy o‘lchovlar orasidagi eng katta farq (shovqin): 6.3%


## 2. Python (numpy) realizatsiyasi

Bir vaqtda 2000 ta mustaqil blok uchun bitta iteratsiya (10 raund) vaqti o‘lchanadi.
Mutlaq qiymatlar C dan ancha sekin, lekin nisbiy farqlar ko‘rsatkichli.

In [6]:
NATIJA['Python'] = {}
rng = np.random.default_rng(1); N = 500 if TEZ else 2000
for p in (256, 16):
    k = kalitlar(rng, N, p); h = rng.integers(0, p, (N, 32)); bl = rng.integers(0, p, (N, 32)); asos = None
    for nom, bz, _, toq, sq in VAR:
        c = Cfg(p, bz); t_ = []
        for r in range(3):
            t0 = time.perf_counter(); iteration_v(h, bl, k.copy(), k, c, bool(toq), bool(sq)); t_.append(time.perf_counter() - t0)
        t = min(t_); asos = asos or t
        NATIJA['Python'][f'{p}_{nom}'] = dict(vaqt_s=round(t, 3), farq_foiz=round((t / asos - 1) * 100, 1))
        print(f"p={p:3d} {nom:10s} {t:.3f} s / {N} blok   farq: {(t/asos-1)*100:+.1f}%")

p=256 std        0.257 s / 2000 blok   farq: +0.0%
p=256 std+toq    0.238 s / 2000 blok   farq: -7.4%
p=256 std+S      0.256 s / 2000 blok   farq: -0.5%
p=256 sbz        0.259 s / 2000 blok   farq: +0.7%
p=256 sbz+toq    0.241 s / 2000 blok   farq: -6.3%
p=256 sbz+toq+S  0.239 s / 2000 blok   farq: -7.1%
p= 16 std        0.267 s / 2000 blok   farq: +0.0%
p= 16 std+toq    0.251 s / 2000 blok   farq: -5.8%
p= 16 std+S      0.275 s / 2000 blok   farq: +3.1%
p= 16 sbz        0.272 s / 2000 blok   farq: +2.0%
p= 16 sbz+toq    0.254 s / 2000 blok   farq: -4.9%
p= 16 sbz+toq+S  0.254 s / 2000 blok   farq: -4.8%


## 3. Amallar sonining modeli

Bitta kirish bloki (bitta iteratsiya) uchun amallar soni C realizatsiyasi tuzilishi bo‘yicha sanaladi:
Qo‘sh — 22 marta, Aralash — 10, SurHolat/SurKalit — 10, TuzilmaKalit — 10, yutish — 3, S-box qatlami — 10.

In [7]:
def amallar(p, bz, toq, sq):
    q = dict(kopaytirish=128 + 64 * 3 + 2 + 64 * 6, teskari_jadval=64 * 3, qoshish=128 + 64 * 6 + 64 * 10)   # bitta Qo‘sh
    a = dict(kopaytirish=128, qoshish=144)                                                                    # bitta Aralash
    n = dict(kopaytirish=22 * q['kopaytirish'] + 10 * a['kopaytirish'], teskari_jadval=22 * q['teskari_jadval'],
             qoshish=22 * q['qoshish'] + 10 * a['qoshish'], jadval=0, taqqoslash=0, bit_amal=0)
    if p == 256: n['jadval'] += 3 * 64; n['bit_amal'] += 3 * 32           # BZ ham, SBZ ham: bayt boshiga 2 ta jadval
    else:
        if bz == 'std': n['bit_amal'] += 3 * 32                            # XOR
        else: n['jadval'] += 3 * 32; n['bit_amal'] += 3 * 64               # 16 bayt x 2 jadval, juftlash
    if toq: n['bit_amal'] += 10 * 32                                       # kₑ ∨ 1
    else: n['taqqoslash'] += 10 * 32 * 2                                   # juftlik va nol tekshiruvi
    if sq: n['jadval'] += 10 * 32                                          # S-box qatlami
    n['jami'] = sum(n.values()); return n
NATIJA['model'] = {}
for p in (256, 16):
    asos = amallar(p, 'std', 0, 0)['jami']
    for nom, bz, _, toq, sq in VAR:
        n = amallar(p, bz, toq, sq); n['farq_foiz'] = round((n['jami'] / asos - 1) * 100, 2); NATIJA['model'][f'{p}_{nom}'] = n
        print(f"p={p:3d} {nom:10s} ko‘paytirish {n['kopaytirish']}, teskarilash jadvali {n['teskari_jadval']}, qo‘shish {n['qoshish']}, "
              f"jadval {n['jadval']}, taqqoslash {n['taqqoslash']}, bit amali {n['bit_amal']}  -> jami {n['jami']} ({n['farq_foiz']:+.2f}%)")

p=256 std        ko‘paytirish 16812, teskarilash jadvali 4224, qo‘shish 26784, jadval 192, taqqoslash 640, bit amali 96  -> jami 48748 (+0.00%)
p=256 std+toq    ko‘paytirish 16812, teskarilash jadvali 4224, qo‘shish 26784, jadval 192, taqqoslash 0, bit amali 416  -> jami 48428 (-0.66%)
p=256 std+S      ko‘paytirish 16812, teskarilash jadvali 4224, qo‘shish 26784, jadval 512, taqqoslash 640, bit amali 96  -> jami 49068 (+0.66%)
p=256 sbz        ko‘paytirish 16812, teskarilash jadvali 4224, qo‘shish 26784, jadval 192, taqqoslash 640, bit amali 96  -> jami 48748 (+0.00%)
p=256 sbz+toq    ko‘paytirish 16812, teskarilash jadvali 4224, qo‘shish 26784, jadval 192, taqqoslash 0, bit amali 416  -> jami 48428 (-0.66%)
p=256 sbz+toq+S  ko‘paytirish 16812, teskarilash jadvali 4224, qo‘shish 26784, jadval 512, taqqoslash 0, bit amali 416  -> jami 48748 (+0.00%)
p= 16 std        ko‘paytirish 16812, teskarilash jadvali 4224, qo‘shish 26784, jadval 0, taqqoslash 640, bit amali 96  -> jami 48556 (+0.00

## 4. Jamlanma

In [8]:
print(f"{'rejim':>6s} {'variant':>11s} {'C, %':>7s} {'Python, %':>10s} {'model, %':>9s}")
for p in (256, 16):
    for nom, *_ in VAR:
        cf = NATIJA.get('C', {}).get(f'{p}_{nom}', {}).get('farq_foiz', float('nan'))
        print(f"{p:6d} {nom:>11s} {cf:+7.1f} {NATIJA['Python'][f'{p}_{nom}']['farq_foiz']:+10.1f} {NATIJA['model'][f'{p}_{nom}']['farq_foiz']:+9.2f}")
json.dump(NATIJA, open('tezlik_natijalari.json', 'w'), ensure_ascii=False, indent=1)
print("Natijalar tezlik_natijalari.json ga yozildi.")
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    for i, p in enumerate((256, 16)):
        noms = [v[0] for v in VAR]; x = np.arange(len(noms))
        for j, (manba, lab) in enumerate([('C', 'C (gcc -O2)'), ('Python', 'Python (numpy)'), ('model', 'amallar modeli')]):
            if manba not in NATIJA: continue
            ax[i].bar(x + (j - 1) * 0.27, [NATIJA[manba][f'{p}_{n}']['farq_foiz'] for n in noms], 0.27, label=lab)
        ax[i].set_xticks(x); ax[i].set_xticklabels(noms, rotation=30, fontsize=8); ax[i].axhline(0, color='k', lw=.6)
        ax[i].set_title(f'p = {p}'); ax[i].set_ylabel('standartga nisbatan farq, %'); ax[i].grid(alpha=.3)
    ax[0].legend(fontsize=8); fig.tight_layout(); fig.savefig('tezlik.png', dpi=200); plt.show()
except ImportError:
    print("matplotlib o‘rnatilmagan — grafik chizilmadi.")

 rejim     variant    C, %  Python, %  model, %
   256         std    +0.0       +0.0     +0.00
   256     std+toq    -0.9       -7.4     -0.66
   256       std+S    +1.9       -0.5     +0.66
   256         sbz    +0.8       +0.7     +0.00
   256     sbz+toq    -0.5       -6.3     -0.66
   256   sbz+toq+S    +1.1       -7.1     +0.00
    16         std    +0.0       +0.0     +0.00
    16     std+toq    +1.1       -5.8     -0.66
    16       std+S    +1.3       +3.1     +0.66
    16         sbz    -0.5       +2.0     +0.40
    16     sbz+toq    +0.5       -4.9     -0.26
    16   sbz+toq+S    +1.9       -4.8     +0.40
Natijalar tezlik_natijalari.json ga yozildi.
